# MatMul Bench v4 — React + Apache ECharts Web UI trên Google Colab Pro

Giao diện full-width responsive (React 18 + Apache ECharts 5), có **Live Split (cột Thời gian tăng dần theo thời gian thực + GFLOPS nảy tức thì)**, **chế độ phóng to biểu đồ toàn màn hình (`F` / `Esc`)**, **3 Preset Demo 1-click** và **đồ họa 2 tầng Hybrid CPU–GPU (`α*` + Idle Wait)**.

> Chọn **Runtime → Change runtime type → GPU (T4 / L4 / A100)** rồi chạy lần lượt 3 cell dưới đây.

## Bước 1: Clone/Pull repo & tự nhận kiến trúc GPU để biên dịch CUDA

In [ ]:
import os, subprocess

if not os.path.exists('/content/Matrix_Multiplication'):
    !git clone https://github.com/dthai2424/Matrix_Multiplication.git /content/Matrix_Multiplication
else:
    !git -C /content/Matrix_Multiplication pull origin main

%cd /content/Matrix_Multiplication
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!echo "CPU vCores: $(nproc)"

cap = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], text=True
).strip().splitlines()[0].replace('.', '')
cuda_arch = f'sm_{cap}'
print(f'>>> Biên dịch CUDA_ENABLED=1 CUDA_ARCH={cuda_arch}')
!make clean && make CUDA_ENABLED=1 CUDA_ARCH={cuda_arch}

## Bước 2: Cài FastAPI & Uvicorn (không cần cài Node.js — bản build React+ECharts đã nằm sẵn trong `web_ui/dist/`)

In [ ]:
!pip install -q fastapi uvicorn
!ls -lh /content/Matrix_Multiplication/web_ui/dist /content/Matrix_Multiplication/web_ui/dist/assets

## Bước 3: Khởi chạy Web UI & mở link HTTPS công khai (Cloudflare Tunnel)

Mở đường link `https://xxxx.trycloudflare.com` in ra bên dưới. Giữ cell này chạy trong suốt lúc thuyết trình.

In [ ]:
%cd /content/Matrix_Multiplication
!MATMUL_REPO_DIR=/content/Matrix_Multiplication python web_ui/colab_tunnel.py

## Bước 4 (Tùy chọn): Tải toàn bộ file kết quả về máy sau khi tắt tunnel

In [ ]:
from google.colab import files
import os

for fname in ['benchmark_results.xls', 'benchmark_results.csv', 'benchmark_charts.html', 'hybrid_alpha.cfg']:
    path = os.path.join('/content/Matrix_Multiplication', fname)
    if os.path.exists(path):
        files.download(path)